# Module A.3: Structured Output
**Part II — Applied LLM Engineering**

> Getting reliable machine-readable output from a model that only speaks prose.

## 1. Why Structured Output?

LLMs are trained to produce natural language. Ask a question, get prose back. That is great for a chatbot — but nearly every real application needs to *do something* with the model's answer: store it in a database, feed it to another API, render it in a UI, or pass it to a downstream function.

Prose is unusable for those tasks. You need **machine-readable output**: JSON objects, typed fields, validated schemas.

This module builds the full structured output stack from scratch:
1. Prompt the model to emit JSON (the simplest fix)
2. Robustly parse the output
3. Validate the result against a schema
4. Retry intelligently on failure
5. Replace your hand-rolled code with Pydantic
6. Understand constrained decoding for the cases where prompting is not reliable enough

Each step is built from scratch before pointing to the production library that already does it.

### The problem in one cell

Ask a plain factual question and watch what you actually get back.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import json
import re

MODEL = "HuggingFaceTB/SmolLM2-135M-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
llm = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float32)

def chat(messages, max_new_tokens=128, temperature=0.0):
    inputs = tok.apply_chat_template(
        messages, add_generation_prompt=True,
        return_tensors="pt", return_dict=True
    )
    do_sample = bool(temperature and temperature > 0)
    out = llm.generate(
        **inputs, max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=(temperature if do_sample else None),
        pad_token_id=tok.eos_token_id
    )
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

# Ask a plain question — observe freeform prose
response = chat([
    {"role": "user", "content": "What is the capital of France?"}
])
print("Raw model output:")
print(repr(response))
print()

# What we actually want
print("What we need for a downstream API call:")
print('{"capital": "Paris"}')
print()
print("Attempting json.loads on the raw output...")
try:
    parsed = json.loads(response)
    print("Parsed:", parsed)
except json.JSONDecodeError as e:
    print(f"FAILED: {e}")
    print("This is the core problem — prose is not JSON.")

## 2. Prompt Engineering for JSON

The fastest fix: tell the model exactly what format you want in the system prompt. This works surprisingly often and costs zero extra compute.

The key ingredients:
- A system prompt that mandates JSON and describes the exact keys
- An instruction to return *only* the JSON object — no preamble, no explanation
- A concrete example in the prompt (few-shot) helps small models a lot

The limitation: the model may still add text before or after the JSON, especially under temperature > 0, or when it does not fully follow the instruction. We will handle that in Section 3.

In [ ]:
SYSTEM_JSON = """\
You are a structured data extractor. 
Always respond with a single JSON object and nothing else — no explanation, no prose, no markdown fences.
Example format: {"capital": "...", "country": "..."}
"""

def ask_for_json(user_message, max_new_tokens=64):
    return chat(
        [
            {"role": "system", "content": SYSTEM_JSON},
            {"role": "user",   "content": user_message},
        ],
        max_new_tokens=max_new_tokens,
        temperature=0.0,  # greedy: most deterministic
    )

# Try the capital question again with the JSON instruction
output = ask_for_json("What is the capital of France? Return JSON with keys: capital, country.")
print("Model output (with JSON prompt):")
print(repr(output))
print()

# Try a second example
output2 = ask_for_json("What is the boiling point of water in Celsius and Fahrenheit? "
                        "Return JSON with keys: celsius, fahrenheit.")
print("Second example output:")
print(repr(output2))

### Reliability issues

Try a more complex extraction. Observe that the output may:
- Include a markdown code fence (` ```json ... ``` `)
- Prepend a sentence like "Here is the JSON:"
- Trail off with an explanation after the closing brace

This is why prompting alone is not enough — we need a robust parser.

In [ ]:
# Simulate the kinds of messy outputs that LLMs produce even with a JSON prompt.
# These are real patterns observed in production.
messy_outputs = [
    '{"capital": "Paris", "country": "France"}',           # clean
    'Sure! Here you go: {"capital": "Paris", "country": "France"}',  # prose prefix
    '```json\n{"capital": "Paris", "country": "France"}\n```',        # markdown fence
    '{"capital": "Paris", "country": "France"} Hope that helps!',     # prose suffix
    '{"capital": "Paris",\n  "country": "France"\n}',                 # pretty-printed
]

print("Attempting json.loads directly on each:")
for i, text in enumerate(messy_outputs):
    try:
        result = json.loads(text)
        print(f"  [{i}] OK    -> {result}")
    except json.JSONDecodeError as e:
        print(f"  [{i}] FAIL  -> {e.msg}")

## 3. Parsing and Extraction

We need a two-stage parser:
1. Try the fast path: `json.loads` directly. Works for clean output.
2. Fall back to a regex that extracts the JSON object or array from surrounding text.

The regex strategy: find the first `{` and its matching `}`, treating the content between them as the JSON payload. We use `re.DOTALL` so `.` matches newlines (pretty-printed JSON).

In [ ]:
def parse_json(text: str) -> dict:
    """Extract and parse a JSON object from an LLM response.

    Strategy:
    1. Fast path: json.loads(text.strip()) — works when output is clean.
    2. Regex fallback: find the first {...} block, handling nested braces
       by scanning for balanced delimiters.

    Raises ValueError if no JSON object can be found.
    """
    # Fast path
    text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # Strip markdown code fences first
    # Pattern: ```json ... ``` or ``` ... ```
    fence_match = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
    if fence_match:
        try:
            return json.loads(fence_match.group(1))
        except json.JSONDecodeError:
            pass

    # Balanced-brace extraction: find the span from the first '{' to its
    # matching '}', correctly handling nested objects.
    start = text.find("{")
    if start == -1:
        raise ValueError(f"No JSON object found in: {text!r}")

    depth = 0
    in_string = False
    escape_next = False
    for i, ch in enumerate(text[start:], start=start):
        if escape_next:
            escape_next = False
            continue
        if ch == "\\" and in_string:
            escape_next = True
            continue
        if ch == '"':
            in_string = not in_string
        if not in_string:
            if ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    candidate = text[start : i + 1]
                    try:
                        return json.loads(candidate)
                    except json.JSONDecodeError as e:
                        raise ValueError(
                            f"Found JSON-like text but it did not parse: {e}"
                        ) from e

    raise ValueError(f"Unbalanced braces in: {text!r}")


# Test against all the messy cases from Section 2
print("Testing parse_json on messy outputs:")
for i, text in enumerate(messy_outputs):
    try:
        result = parse_json(text)
        print(f"  [{i}] OK    -> {result}")
    except ValueError as e:
        print(f"  [{i}] FAIL  -> {e}")

# Test the failure case explicitly
print()
try:
    parse_json("The capital of France is Paris.")
except ValueError as e:
    print(f"No JSON case: {e}")

## 4. Schema Validation From Scratch

Parsing succeeds when the JSON is well-formed. But the JSON could be *valid syntax* and still wrong for our application — missing a required key, a value of the wrong type, an empty string where we need a name.

We need **schema validation**: a function that checks the parsed dict satisfies our contract.

We will build a minimal validator that handles:
- Required keys
- Type checking (`str`, `int`, `float`, `bool`, `list`)

This is intentionally simple. In Section 6 we will replace it with Pydantic, which handles the full JSON Schema spec. The point of building it by hand is to understand *what* the library is doing.

In [ ]:
def validate_schema(data: dict, schema: dict) -> None:
    """Validate `data` against a simple schema definition.

    Schema format:
        {
          "field_name": {"type": <python type>, "required": True/False},
          ...
        }

    Raises ValueError with a descriptive message on the first violation.
    Returns None on success.
    """
    for field, spec in schema.items():
        required = spec.get("required", True)
        expected_type = spec["type"]

        if field not in data:
            if required:
                raise ValueError(f"Missing required field: '{field}'")
            continue  # optional field absent — that is fine

        value = data[field]

        # Allow int where float is expected (Python: int is not a subclass of float)
        if expected_type is float and isinstance(value, int):
            continue

        if not isinstance(value, expected_type):
            raise ValueError(
                f"Field '{field}': expected {expected_type.__name__}, "
                f"got {type(value).__name__} ({value!r})"
            )


# --- Define a schema for a geography fact ---
CAPITAL_SCHEMA = {
    "capital":    {"type": str,  "required": True},
    "country":    {"type": str,  "required": True},
    "population": {"type": int,  "required": False},
}

# Valid data
valid_data = {"capital": "Paris", "country": "France", "population": 2161000}
validate_schema(valid_data, CAPITAL_SCHEMA)
print("Valid data passed:", valid_data)

# Valid without the optional field
valid_minimal = {"capital": "Berlin", "country": "Germany"}
validate_schema(valid_minimal, CAPITAL_SCHEMA)
print("Minimal data passed:", valid_minimal)
print()

# Invalid cases
invalid_cases = [
    {"capital": "Paris"},                                   # missing required 'country'
    {"capital": 42, "country": "France"},                    # 'capital' is int, not str
    {"capital": "Paris", "country": "France",
     "population": "two million"},                          # 'population' wrong type
]

print("Invalid cases (each should raise ValueError):")
for i, bad in enumerate(invalid_cases):
    try:
        validate_schema(bad, CAPITAL_SCHEMA)
        print(f"  [{i}] UNEXPECTED PASS for {bad}")
    except ValueError as e:
        print(f"  [{i}] Caught correctly: {e}")

## 5. Retry Loop

Parsing and validation can fail. The model might emit malformed JSON, miss a field, or use the wrong type. The right response is not to crash — it is to tell the model what went wrong and try again.

The key insight: **the error message is the new prompt**. Append it to the conversation and re-generate. The model can see what it got wrong and self-correct.

This is the structured output retry pattern used in virtually every production LLM pipeline.

In [ ]:
def call_with_schema(
    messages: list,
    schema: dict,
    max_retries: int = 3,
    max_new_tokens: int = 128,
) -> dict:
    """Call the LLM and return a validated dict, retrying on parse/validation failure.

    On each failure the model's bad response and the error message are appended
    to the conversation so the model can see and correct its mistake.

    Raises RuntimeError if all attempts are exhausted.
    """
    # Work on a copy so we don't mutate the caller's list
    convo = list(messages)

    for attempt in range(1, max_retries + 1):
        raw = chat(convo, max_new_tokens=max_new_tokens, temperature=0.0)
        print(f"  Attempt {attempt}: {raw!r}")

        # --- Step 1: parse ---
        try:
            data = parse_json(raw)
        except ValueError as parse_err:
            error_msg = f"Your output could not be parsed as JSON. Error: {parse_err}. Please return only a JSON object."
            convo.append({"role": "assistant", "content": raw})
            convo.append({"role": "user",      "content": error_msg})
            print(f"           -> Parse error, retrying. ({parse_err})")
            continue

        # --- Step 2: validate ---
        try:
            validate_schema(data, schema)
        except ValueError as val_err:
            error_msg = (
                f"Your JSON was parsed but failed schema validation: {val_err}. "
                f"Please fix the output and return only the corrected JSON object."
            )
            convo.append({"role": "assistant", "content": raw})
            convo.append({"role": "user",      "content": error_msg})
            print(f"           -> Validation error, retrying. ({val_err})")
            continue

        # Both steps passed
        print(f"           -> Success on attempt {attempt}.")
        return data

    raise RuntimeError(
        f"Failed to obtain valid structured output after {max_retries} attempts."
    )


# --- Demo: extract a geography fact ---
SYSTEM_EXTRACTOR = """\
You are a structured data extractor.
Always respond with a single JSON object only — no prose, no markdown fences.
Required keys: capital (string), country (string).
"""

CAPITAL_SCHEMA_STRICT = {
    "capital": {"type": str, "required": True},
    "country": {"type": str, "required": True},
}

print("--- Extraction with retry loop ---")
try:
    result = call_with_schema(
        messages=[
            {"role": "system", "content": SYSTEM_EXTRACTOR},
            {"role": "user",   "content": "What is the capital of Japan? Return JSON with keys: capital, country."},
        ],
        schema=CAPITAL_SCHEMA_STRICT,
        max_retries=3,
    )
    print("\nFinal validated result:", result)
except RuntimeError as e:
    # A 135M model is often too weak to satisfy the schema even after retries.
    # We catch the exhaustion so the notebook still runs end to end -- and because
    # the *failure* is itself a lesson (see the note below), not a bug in our code.
    print(f"\n{e}")
    print("(Expected with a tiny model -- see the note below. The retry PATTERN is")
    print(" still exactly right; the next cell proves it recovers deterministically.)")

> ⚠️ **On the tiny demo model.** The labs default to `SmolLM2-135M-Instruct` so
> they run free on a CPU — but a 135M-parameter model is genuinely *weak*, and at
> `temperature=0` it will often answer a "return JSON" request with plain prose
> ("The capital of Japan is Tokyo."), exhausting the retry budget. **That is not a
> bug in the pipeline — it's the honest limit of a small model,** and it's exactly
> why production systems reach for (a) a stronger model, (b) few-shot examples, or
> (c) *constrained decoding* (Section 7), which makes invalid JSON impossible
> rather than merely discouraged. The cells here catch that exhaustion so the
> notebook always runs; the next cell demonstrates the retry loop **succeeding**
> deterministically, so you see the mechanism work regardless of the live model.

In [ ]:
# Demonstrate recovery from a parse failure by injecting a deliberately broken
# first response into the conversation, then letting the retry loop correct it.
#
# We monkey-patch `chat` temporarily to return bad output on the first call
# and good output on subsequent calls.

original_chat = chat
call_count = [0]

def flaky_chat(messages, max_new_tokens=128, temperature=0.0):
    call_count[0] += 1
    if call_count[0] == 1:
        # Intentionally bad: prose with no JSON at all
        return "The capital of Germany is Berlin, a vibrant city."
    # Second call: the model has seen the error and returns correct JSON
    return '{"capital": "Berlin", "country": "Germany"}'

import builtins
# Temporarily replace the chat function for the test
import types

# We need to patch the local 'chat' name inside call_with_schema.
# The simplest approach: redefine call_with_schema to accept a chat_fn parameter.

def call_with_schema_v2(
    messages: list,
    schema: dict,
    chat_fn=None,
    max_retries: int = 3,
    max_new_tokens: int = 128,
) -> dict:
    """Same as call_with_schema but accepts a pluggable chat function (useful for testing)."""
    if chat_fn is None:
        chat_fn = chat
    convo = list(messages)
    for attempt in range(1, max_retries + 1):
        raw = chat_fn(convo, max_new_tokens=max_new_tokens, temperature=0.0)
        print(f"  Attempt {attempt}: {raw!r}")
        try:
            data = parse_json(raw)
        except ValueError as parse_err:
            convo.append({"role": "assistant", "content": raw})
            convo.append({"role": "user",      "content": f"Parse error: {parse_err}. Return only JSON."})
            print(f"           -> Parse error, retrying.")
            continue
        try:
            validate_schema(data, schema)
        except ValueError as val_err:
            convo.append({"role": "assistant", "content": raw})
            convo.append({"role": "user",      "content": f"Validation error: {val_err}. Fix and return only JSON."})
            print(f"           -> Validation error, retrying.")
            continue
        print(f"           -> Success on attempt {attempt}.")
        return data
    raise RuntimeError(f"Failed after {max_retries} attempts.")


print("--- Simulated flaky model: bad output on attempt 1, correct on attempt 2 ---")
call_count[0] = 0
result = call_with_schema_v2(
    messages=[
        {"role": "system", "content": SYSTEM_EXTRACTOR},
        {"role": "user",   "content": "What is the capital of Germany?"},
    ],
    schema=CAPITAL_SCHEMA_STRICT,
    chat_fn=flaky_chat,
    max_retries=3,
)
print("\nFinal result after recovery:", result)

## 6. Pydantic for Production

The hand-rolled `validate_schema` function works, but it only handles the basics. Real schemas need:
- Nested objects and arrays
- String constraints (`min_length`, `regex`)
- Integer ranges (`ge`, `le`)
- Optional fields with defaults
- Detailed, structured error messages
- Auto-generated JSON Schema for documentation

That is **Pydantic**. A `BaseModel` subclass *is* your schema: field annotations declare types, validators declare constraints, and `model_validate()` is your `validate_schema()` call.

The pattern is identical to what you just built — you are replacing the implementation, not the idea.

In [ ]:
from pydantic import BaseModel, field_validator
from typing import Optional

# --- Production schema: replace validate_schema + hand-rolled types ---

class CapitalFact(BaseModel):
    capital: str
    country: str
    population: Optional[int] = None  # optional with default None

    @field_validator("capital", "country")
    @classmethod
    def must_not_be_empty(cls, v: str) -> str:
        if not v.strip():
            raise ValueError("Field must not be empty")
        return v


# Valid case
fact = CapitalFact.model_validate({"capital": "Paris", "country": "France", "population": 2161000})
print("Valid model:", fact)
print("Serialized:", fact.model_dump())
print()

# Optional field absent — fine
fact2 = CapitalFact.model_validate({"capital": "Tokyo", "country": "Japan"})
print("Without population:", fact2)
print()

# Invalid cases — Pydantic gives detailed errors
from pydantic import ValidationError

bad_inputs = [
    {"capital": "Paris"},                             # missing required 'country'
    {"capital": "", "country": "France"},              # empty string caught by validator
    {"capital": "Paris", "country": "France",
     "population": "two million"},                    # wrong type
]

print("Invalid cases:")
for i, bad in enumerate(bad_inputs):
    try:
        CapitalFact.model_validate(bad)
        print(f"  [{i}] UNEXPECTED PASS")
    except ValidationError as e:
        # Show only the first error for brevity
        first_err = e.errors()[0]
        print(f"  [{i}] Caught: field='{first_err['loc'][0]}' msg='{first_err['msg']}'")

In [ ]:
# --- Full production pipeline with Pydantic ---
# Replace validate_schema with model_validate; everything else stays the same.

def call_with_pydantic(
    messages: list,
    model_class: type,       # a Pydantic BaseModel subclass
    chat_fn=None,
    max_retries: int = 3,
    max_new_tokens: int = 128,
):
    """Structured output pipeline backed by a Pydantic model."""
    if chat_fn is None:
        chat_fn = chat
    convo = list(messages)

    for attempt in range(1, max_retries + 1):
        raw = chat_fn(convo, max_new_tokens=max_new_tokens, temperature=0.0)
        print(f"  Attempt {attempt}: {raw!r}")
        try:
            data = parse_json(raw)
        except ValueError as e:
            convo.append({"role": "assistant", "content": raw})
            convo.append({"role": "user",      "content": f"JSON parse error: {e}. Return only a JSON object."})
            print(f"           -> Parse error, retrying.")
            continue
        try:
            instance = model_class.model_validate(data)
        except ValidationError as e:
            # Surface structured error messages back to the model
            error_summary = "; ".join(
                f"{'.'.join(str(x) for x in err['loc'])}: {err['msg']}"
                for err in e.errors()
            )
            convo.append({"role": "assistant", "content": raw})
            convo.append({"role": "user",      "content": f"Validation errors: {error_summary}. Please fix and return only the corrected JSON."})
            print(f"           -> Validation error, retrying. ({error_summary})")
            continue
        print(f"           -> Success on attempt {attempt}.")
        return instance

    raise RuntimeError(f"Failed after {max_retries} attempts.")


# Run the pipeline on a real model call
print("--- Pydantic-backed pipeline ---")
try:
    result = call_with_pydantic(
        messages=[
            {"role": "system", "content": SYSTEM_EXTRACTOR},
            {"role": "user",   "content": "What is the capital of Italy? Return JSON with keys: capital, country."},
        ],
        model_class=CapitalFact,
        max_retries=3,
    )
    print("\nResult type:", type(result))
    print("Result:", result)
    print("capital field:", result.capital)
    print("country field:", result.country)
except RuntimeError as e:
    # Same small-model caveat as the hand-rolled pipeline: the Pydantic swap is
    # about VALIDATION, not about making a weak model comply. On a stronger model
    # (or via constrained decoding, Section 7) this returns a typed CapitalFact.
    print(f"\n{e}")
    print("(Expected with a tiny model. The Pydantic-vs-hand-rolled point stands:")
    print(" model_validate() replaced validate_schema(); everything else is identical.)")

## 7. Constrained Decoding (Concept)

Prompt engineering + retry is good. But there is a more fundamental approach: **constrained decoding**, which makes it *impossible* for the model to generate invalid JSON in the first place.

### How logit masking works

At each generation step, the model computes a logit score for every token in the vocabulary (typically 32 000–128 000 tokens). Normally we just sample or take the argmax.

With constrained decoding, before the argmax or sample step, we set the logit for every *invalid* token to `-inf`. The softmax of `-inf` is 0, so those tokens can never be selected. The model still does the full forward pass — we only intervene at the final logit layer.

```
Model forward pass                     Logit masking
─────────────────    ────────────────────────────────────────
Input tokens   ──►  [hidden states] ──► [logits for all 32k tokens]
                                              │
                          ┌───────────────────┘
                          ▼
                    JSON state machine
                    (knows valid next chars)
                          │
                    set invalid-token logits to -inf
                          │
                          ▼
                    sample/argmax ──► always valid JSON token
```

The JSON state machine tracks what is syntactically valid at each position. After an opening `{`, the only valid tokens are whitespace or a `"` (start of a key). After `"capital":`, only values are valid. And so on.

### When to use it

| Approach | When to use |
|---|---|
| Prompt + parse + retry | Most cases. Zero library overhead. Works with any API. |
| Constrained decoding | When parse failures are still too frequent (small models, complex schemas) or latency for retries is unacceptable. |

### Production libraries

You do not implement this yourself. Two battle-tested options:

- **[Outlines](https://github.com/outlines-dev/outlines)** — framework-agnostic, works with Transformers and vLLM. Supports JSON Schema, regex, and grammar constraints. One function call: `outlines.generate.json(model, MyPydanticModel)`.

- **[Guidance](https://github.com/guidance-ai/guidance)** — from Microsoft. Template-based approach that interleaves generation and logic. More expressive but more opinionated.

Both hook into the exact logit-masking mechanism described above. Their APIs look like this:

```python
# Outlines (concept — do not run this cell, it requires the outlines package)
import outlines

model = outlines.models.transformers("HuggingFaceTB/SmolLM2-135M-Instruct")
generator = outlines.generate.json(model, CapitalFact)  # CapitalFact is our Pydantic model
result = generator("What is the capital of Spain?")
# result is always a valid CapitalFact — no retry needed
```

The tradeoff: constrained decoding adds overhead to the forward pass (state machine evaluation at every token) and only works when you control the model weights. If you are calling an API like Claude or GPT-4, you are back to prompting — which is why the retry loop is still the default workhorse.

## 8. Try It Yourself

Three tasks to solidify the concepts. Each builds on the primitives from earlier sections.

**Task (a)** — Build a structured output pipeline that extracts `{name, age, occupation}` from a bio sentence.

**Task (b)** — Add a retry that includes the *full* validation error message in the prompt. Compare recovery speed vs. a generic "please fix" retry.

**Task (c)** — Try asking for a list field (`skills: list`) and handle the `list` type in your schema validator.

In [ ]:
# Task (a): Bio extraction pipeline
# Extract {name: str, age: int, occupation: str} from a short biography sentence.

BIO_SCHEMA = {
    "name":       {"type": str, "required": True},
    "age":        {"type": int, "required": True},
    "occupation": {"type": str, "required": True},
}

SYSTEM_BIO = """\
You are a structured data extractor.
Given a biography sentence, return a single JSON object with these keys:
  name (string), age (integer), occupation (string).
Return only the JSON object — no prose, no markdown fences.
"""

bio_sentences = [
    "Maria Gonzalez is a 34-year-old software engineer who lives in Barcelona.",
    "James, 52, worked as a marine biologist before retiring to write novels.",
]

for bio in bio_sentences:
    print(f"Bio: {bio}")
    try:
        result = call_with_schema(
            messages=[
                {"role": "system", "content": SYSTEM_BIO},
                {"role": "user",   "content": bio},
            ],
            schema=BIO_SCHEMA,
            max_retries=3,
            max_new_tokens=64,
        )
        print(f"Extracted: {result}\n")
    except RuntimeError as e:
        print(f"  (retries exhausted: {e})")
        print("  A 135M model rarely nails multi-field extraction. Rerun this on a")
        print("  stronger model to see it succeed -- the pipeline code is unchanged.\n")

In [ ]:
# Task (b): Retry with detailed validation error in the prompt
#
# The call_with_schema function above already does this.
# Here, contrast it with a generic retry that does NOT include the error message.
# Observe whether the model self-corrects faster when it knows the exact problem.

call_count_generic = [0]
call_count_detailed = [0]

# Simulate a model that always emits a string age on the first try
def bio_flaky_chat_generic(messages, max_new_tokens=64, temperature=0.0):
    call_count_generic[0] += 1
    if call_count_generic[0] == 1:
        return '{"name": "Alice", "age": "twenty-nine", "occupation": "nurse"}'  # age is string
    return '{"name": "Alice", "age": 29, "occupation": "nurse"}'

def bio_flaky_chat_detailed(messages, max_new_tokens=64, temperature=0.0):
    call_count_detailed[0] += 1
    if call_count_detailed[0] == 1:
        return '{"name": "Alice", "age": "twenty-nine", "occupation": "nurse"}'
    # Model sees the specific error and corrects only the broken field
    return '{"name": "Alice", "age": 29, "occupation": "nurse"}'

# Generic retry: error message just says "please fix"
def call_with_generic_retry(messages, schema, chat_fn, max_retries=3):
    convo = list(messages)
    for attempt in range(1, max_retries + 1):
        raw = chat_fn(convo)
        print(f"  [Generic]  Attempt {attempt}: {raw!r}")
        try:
            data = parse_json(raw)
            validate_schema(data, schema)
            print(f"             -> Success on attempt {attempt}.")
            return data
        except (ValueError,) as e:
            # Generic message — no specific error included
            convo.append({"role": "assistant", "content": raw})
            convo.append({"role": "user",      "content": "Your output was invalid. Please return only valid JSON."})
            print(f"             -> Error (hidden from model), retrying.")
    raise RuntimeError("Failed.")

BASE_MESSAGES = [
    {"role": "system", "content": SYSTEM_BIO},
    {"role": "user",   "content": "Alice is a 29-year-old nurse."},
]

print("=== Generic retry (error hidden from model) ===")
call_count_generic[0] = 0
call_with_generic_retry(BASE_MESSAGES, BIO_SCHEMA, bio_flaky_chat_generic)

print()
print("=== Detailed retry (error shown to model) ===")
call_count_detailed[0] = 0
call_with_schema_v2(BASE_MESSAGES, BIO_SCHEMA, chat_fn=bio_flaky_chat_detailed)

# Both should recover in 2 attempts with this simple flaky model.
# With a real model and complex schemas, the detailed error message often
# reduces the number of retries needed.

In [ ]:
# Task (c): Handle a list field
# Extract {name: str, skills: list} from a profile sentence.

SKILLS_SCHEMA = {
    "name":   {"type": str,  "required": True},
    "skills": {"type": list, "required": True},
}

# validate_schema already handles list (isinstance check works for list)
# Let's verify:
test_cases = [
    ({"name": "Eve", "skills": ["Python", "SQL"]}, "valid"),
    ({"name": "Eve", "skills": "Python, SQL"},     "invalid — skills is string not list"),
    ({"name": "Eve"},                              "invalid — missing skills"),
]

print("Schema validation with list field:")
for data, desc in test_cases:
    try:
        validate_schema(data, SKILLS_SCHEMA)
        print(f"  PASS ({desc}): {data}")
    except ValueError as e:
        print(f"  FAIL ({desc}): {e}")

print()

# Now run the full pipeline
SYSTEM_SKILLS = """\
You are a structured data extractor.
Given a professional profile, return a JSON object with keys:
  name (string), skills (array of strings).
Return only the JSON object — no prose, no markdown fences.
Example: {"name": "Alice", "skills": ["Python", "SQL", "Machine Learning"]}
"""

profile = "David Chen is a full-stack developer proficient in React, Node.js, and PostgreSQL."
print(f"Profile: {profile}")
result = call_with_schema(
    messages=[
        {"role": "system", "content": SYSTEM_SKILLS},
        {"role": "user",   "content": profile},
    ],
    schema=SKILLS_SCHEMA,
    max_retries=3,
    max_new_tokens=80,
)
print(f"\nExtracted:")
print(f"  name:   {result['name']}")
print(f"  skills: {result['skills']}")
print(f"  skills is a list: {isinstance(result['skills'], list)}")

---

## Summary

You built the structured output stack layer by layer:

| Layer | What you built | Production equivalent |
|---|---|---|
| Prompting | System prompt mandating JSON | Same — no library needed |
| Parsing | `parse_json()`: direct load → regex fallback | Same pattern, used everywhere |
| Validation | `validate_schema()`: required keys + types | **Pydantic** `BaseModel.model_validate()` |
| Retry | `call_with_schema()`: parse → validate → retry with error in prompt | Same pattern; your Pydantic version is production-ready |
| Constrained decoding | Concept: logit masking via JSON state machine | **Outlines** / **Guidance** |

**The mental model**: prompt first, parse robustly, validate against schema, retry with the specific error. If retries are still failing, constrained decoding guarantees structural validity at the cost of requiring local model access.

**Next**: Module A.4 — RAG (Retrieval-Augmented Generation): giving the model access to external knowledge at inference time.